# TURN-Mapper: CAD line classifier from FloorPlanCAD

Trains `src/turn_mapper/primitives.py` (wall / door / window / furniture / stairs per drawn line) on FloorPlanCAD (Fan et al., ICCV 2021; annotations CC BY-NC 4.0, drawings not owned by the authors; non-commercial research use). Then re-runs the 28 public plans with the classifier dropping furniture lines. Internet: On. Accelerator: none needed. Data and model stay in /kaggle/working and are never committed.

In [ ]:
!git clone -q https://github.com/ZakiNabeel/Turn-Mapper.git /kaggle/working/Turn-Mapper
%cd /kaggle/working/Turn-Mapper
!apt-get -qq install -y tesseract-ocr > /dev/null 2>&1; pip -q install -e . svgpathtools gdown 2>&1 | tail -1; git log --oneline -1
import os, subprocess, shutil, glob
D = "/kaggle/working/fpc"; os.makedirs(D, exist_ok=True)
ids = {"train1": "1HcyKt6qWeXog-tRfvEjdO3O3TN91PXGL", "train2": "1kSS7OB_EEu7VJzb0W8DK9_nu1EvshioV", "test": "1jxpYgxnLUbXEzMOsjaMPQFSuvmvHimiZ"}
for name, fid in ids.items():
    out = f"{D}/{name}.archive"
    subprocess.run(["gdown", "-q", "--fuzzy", f"https://drive.google.com/file/d/{fid}/view", "-O", out])
    if not os.path.exists(out):
        print("FETCH FAIL", name); continue
    print("FETCHED", name, os.path.getsize(out))
    os.makedirs(f"{D}/{name}", exist_ok=True)
    for fmt in ("zip", "gztar", "xztar", "tar"):
        try:
            shutil.unpack_archive(out, f"{D}/{name}", fmt); break
        except Exception:
            continue
    os.remove(out)
    print("SVGS", name, len(glob.glob(f"{D}/{name}/**/*.svg", recursive=True)))
!python scripts/floorplancad_train.py --train /kaggle/working/fpc/train1 /kaggle/working/fpc/train2 --test /kaggle/working/fpc/test --out /kaggle/working/fpc_lines.joblib --max-train 4000 --max-test 1000 2>&1 | grep -E "^(TRAIN|TEST|SAVED|SKIP)" | head -40
!rm -rf /kaggle/working/fpc
!TURN_LINE_MODEL=/kaggle/working/fpc_lines.joblib python scripts/kaggle_public_eval.py --configs classical --weights none --work /kaggle/working/public_eval_lines 2>&1 | grep -E "^(OK|FETCH|NOT|RUN|SUMMARY|ALL DONE|[a-z_0-9]+ +n=)"
